In [16]:
#%pip install scipy
#%pip install statsmodels

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy.stats import spearmanr, pearsonr
import glob

In [18]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
# Percorso della cartella con i CSV puliti
#path = "/content/drive/MyDrive/Desktop/Magistrale/IoT_ESP_SleepSense/Misurazioni/*.csv"
# path = "C:/Users/leopi/Il mio Drive/Desktop/Magistrale/IoT_ESP_SleepSense/Misurazioni/D_*.csv"
path = "C:/Users/leopi/Il mio Drive/Desktop/Magistrale/IoT_ESP_SleepSense/*.csv"

# Caricamento file
all_files = glob.glob(path)
dfs = [pd.read_csv(f) for f in all_files]
df_all = pd.concat(dfs, ignore_index=True)

# Pulizia e conversione tipi
df_all["_time"] = pd.to_datetime(df_all["_time"], utc=True, errors="coerce", format="mixed")
cols_to_numeric = ["humidity", "light", "mic", "temperature", "is_moving"]
df_all[cols_to_numeric] = df_all[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# Rimuovo eventuali righe con valori nulli
df_all = df_all.dropna(subset=cols_to_numeric)

print("Dimensione dataset:", df_all.shape)
df_all.head()

fig = px.line(df_all, x="_time", y=cols_to_numeric, title="Plot delle variabili nel tempo")
fig.update_layout(xaxis_title="Time", yaxis_title="Sensor Values")
fig.show()

Dimensione dataset: (361, 16)


In [20]:
fig = px.violin(df_all, y=cols_to_numeric, box=True, points="all",
                title="Distribuzione dei valori dei sensori (Violin Plot)")
fig.show()

fig = px.line(df_all, y=cols_to_numeric, title="Plot delle variabili affiancate (senza tempo)")
fig.update_layout(xaxis_title="Index", yaxis_title="Sensor Values")
fig.show()


In [21]:
# Definisco due dataframe per le analisi
df1 = df_all.copy() # dataframe originale
df2 = df_all.copy() # dataframe con is_moving binarizzato
df2["is_moving"] = df2["is_moving"].apply(lambda x: 1 if x != 0 else 0)
cols = ["humidity", "light", "mic", "temperature", "is_moving"]

In [22]:
# Matrici di correlazione ===

# Pearson
pear1 = df1[cols].corr(method="pearson")["is_moving"]
pear2 = df2[cols].corr(method="pearson")["is_moving"]

# Spearman
spear1 = df1[cols].corr(method="spearman")["is_moving"]
spear2 = df2[cols].corr(method="spearman")["is_moving"]

# Kendall
kend1 = df1[cols].corr(method="kendall")["is_moving"]
kend2 = df2[cols].corr(method="kendall")["is_moving"]

# === 4. Rimuovo la riga is_moving == se stesso ===
variables = ["humidity", "light", "mic", "temperature"]

pear1 = pear1[variables]
pear2 = pear2[variables]
spear1 = spear1[variables]
spear2 = spear2[variables]
kend1 = kend1[variables]
kend2 = kend2[variables]

# === 5. Creo un dataframe comparativo ===
df_compare = pd.DataFrame({
    "Variabile": variables,
    "Pearson_originale": pear1.values,
    "Pearson_binario": pear2.values,
    "Spearman_originale": spear1.values,
    "Spearman_binario": spear2.values,
    "Kendall_originale": kend1.values,
    "Kendall_binario": kend2.values,
})

print(df_compare)


     Variabile  Pearson_originale  Pearson_binario  Spearman_originale  \
0     humidity           0.160238         0.135919            0.136164   
1        light           0.107070         0.091930            0.093557   
2          mic           0.020068         0.001360           -0.078097   
3  temperature          -0.326149        -0.305430           -0.255317   

   Spearman_binario  Kendall_originale  Kendall_binario  
0          0.124099           0.111372         0.101398  
1          0.089652           0.085583         0.085033  
2         -0.075337          -0.060381        -0.061527  
3         -0.240991          -0.204721        -0.197574  


In [23]:
fig = go.Figure()

for metric, color in [
    ("Pearson", "blue"),
    ("Spearman", "green"),
    ("Kendall", "orange")
]:
    fig.add_trace(go.Bar(
        x=df_compare["Variabile"],
        y=df_compare[f"{metric}_originale"],
        name=f"{metric} (originale)",
        marker=dict(color=color, opacity=0.55)
    ))
    fig.add_trace(go.Bar(
        x=df_compare["Variabile"],
        y=df_compare[f"{metric}_binario"],
        name=f"{metric} (binario)",
        marker=dict(color=color)
    ))

fig.update_layout(
    title="Confronto correlazioni con is_moving (originale vs binario)",
    xaxis_title="Variabile",
    yaxis_title="Correlazione",
    barmode="group",
    yaxis=dict(range=[-1,1])
)

fig.show()

In [24]:

# 3. Scatter plot movimento vs ogni variabile
for col in ["humidity", "light", "mic", "temperature"]:
    fig_scatter = px.scatter(df_all, x=col, y="is_moving", trendline="ols",
                             title=f"Movimento vs {col}")
    fig_scatter.show()
